In [1]:
import torch
torch.cuda.is_available()

d:\CodeFile\intro-to-programming-1-python\Shakespeare LLM\Shakespeare-LLM\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


False

In [4]:
import re
from collections import Counter, defaultdict

class BPETokenizer:
    def __init__(self, vocab_size: int = 10000, end_of_word: str = '</w>'):
        self.vocab_size : int   = vocab_size
        self.end_of_word: str   = end_of_word
        self.merges      : list  = list()
        self.vocab      : set   = set()
        self.char_to_id : dict  = dict()
        self.id_to_char : dict  = dict()
    
    def _get_word_parts(self, word: str) -> list[str]:
        return list(word) + [self.end_of_word]
    
    def _count_pairs(self, corpus: list[list[str]]) -> dict[tuple[str, str], int]:
        pair_counts = defaultdict(int)
        for word in corpus:
            for i in range(len(word)-1):
                pair_counts[word[i], word[i+1]] += 1
        return pair_counts

    def train(self, text: str, suppress: bool = False):
        words = re.findall(r'\S+', text)
        corpus = [self._get_word_parts(w) for w in words]

        chars = set()
        for w in corpus:
            chars.update(w)
        self.vocab = chars
        self.vocab.add(self.end_of_word)

        for num in range(self.vocab_size - len(self.vocab)):
            pair_counts = self._count_pairs(corpus)
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            new_token = best_pair[0] + best_pair[1]
            self.merges.append((best_pair, new_token))
            self.vocab.add(new_token)

            if not suppress:
                print(f"new token {num+1} : {new_token}")

            new_corpus = []
            for word in corpus:
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word)-1 and (word[i], word[i+1]) == best_pair:
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_corpus.append(new_word)
            corpus = new_corpus
        
        self.token_to_id = {tok: i for i, tok in enumerate(sorted(self.vocab))}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}
    
    def encode(self, text: str):
        words = re.findall(r'\S+', text)
        tokens = []
        for w in words:
            parts = self._get_word_parts(w)

            for (a, b), merge in self.merges:
                new_parts = []
                i = 0
                while i < len(parts):
                    if i < len(parts)-1 and parts[i] == a and parts[i+1] == b:
                        new_parts.append(merge)
                        i += 2
                    else:
                        new_parts.append(parts[i])
                        i += 1
                parts = new_parts
            tokens.extend(parts)
        return [self.token_to_id[t] for t in tokens if t in self.token_to_id]
    
    def decode(self, ids: list[int]):
        tokens = [self.id_to_token[i] for i in ids]
        text = ''.join(tokens).replace(self.end_of_word, ' ')
        text = re.sub(r'\s+', ' ', text)
        return text
    
    def save(self, path: str):
        import json
        data = {
            'vocab_size'    : self.vocab_size,
            'end_of_word'   : self.end_of_word,
            'merges'        : [(list(pair), merged) for (pair, merged) in self.merges],
            'vocab'         : list(self.vocab),
            'token_to_id'   : self.token_to_id,
            'id_to_token'   : self.id_to_token,
        }
        with open(path, 'w', encoding='utf-8') as file:
            json.dump(data, file)
    
    def load(self, path: str):
        import json
        with open(path, 'r', encoding='utf-8') as file:
            data = json.load(file)
        self.vocab_size = data['vocab_size']
        self.end_of_word = data['end_of_word']
        self.merges = [(tuple(pair), merged) for (pair, merged) in data['merges']]
        self.vocab = set(data['vocab'])
        self.token_to_id = data['token_to_id']
        self.id_to_token = {int(k): v for k, v in data['id_to_token'].items()}

In [5]:
import os

tokenizer = BPETokenizer(vocab_size=10000)

save_path = 'tokenizer/version_1.json'

with open('dataset/The Complete Works of William Shakespeare.txt', 'r', encoding='utf-8') as file:
    text = file.read()

if os.path.exists(save_path):
    tokenizer.load(save_path)
else:
    tokenizer.train(text)
    tokenizer.save(save_path)


sample = 'to be or not to be'
ids = tokenizer.encode(sample)
print(ids)
print(tokenizer.decode(ids))

[9110, 3353, 7152, 6990, 9110, 3353]
to be or not to be 
